In [ ]:
A script to match the excel spreadsheet of SSM data to the full observational dataset of LiveOcean and SalishSeaCast

In [17]:
import pandas as pd
import numpy as np

In [64]:
# read in ssm data

ssm = pd.read_excel('./ssm_model_obs_pair.xlsx', sheet_name='Sheet1')

In [ ]:
print(ssm)

       Station          Model_Time  Nodes   ID Type  Depth_m Masked_Area?  \
0       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      1.0           No   
1       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      1.5           No   
2       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      2.0           No   
3       ADM001 2014-03-13 12:00:00   6231  MMU  Lab      1.5           No   
4       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      2.5           No   
...        ...                 ...    ...  ...  ...      ...          ...   
101239  FID001 2014-11-18 12:00:00   5830  MMU  CTD      8.0           No   
101240  FID001 2014-11-18 12:00:00   5830  MMU  CTD      8.5           No   
101241  FID001 2014-11-18 12:00:00   5830  MMU  CTD      9.0           No   
101242  FID001 2014-11-18 12:00:00   5830  MMU  CTD     11.5           No   
101243  FID001 2014-11-18 12:00:00   5830  MMU  CTD     12.0           No   

        Temp_C  Salinity_psu  DO_mgL  ...  Model_DO_mgL  Model_NO23N_mgL  \

In [ ]:
ssm.columns

Index(['Station', 'Model_Time', 'Nodes', 'ID', 'Type', 'Depth_m',
       'Masked_Area?', 'Temp_C', 'Salinity_psu', 'DO_mgL', 'Chla_ugL',
       'NO23N_mgL', 'NH4N_mgL', 'PAR_Em2day', 'Layer', 'Model_Temp_C',
       'Model_Salinity_psu', 'Model_DO_mgL', 'Model_NO23N_mgL',
       'Model_NH4N_mgL', 'Model_PAR_Em2day', 'Model_Chla_ugL', 'Layer_Depth_m',
       'Embayment', 'Layer_Cat', 'Hydro_T', 'WQM_T'],
      dtype='object')

In [ ]:
# bottle samples are where there is a measurement of NO23N_mgL
print(ssm['NO23N_mgL'].notna().sum())

1916


In [65]:
# keep only bottle data
ssm = ssm.dropna(subset=['NO23N_mgL']).reset_index(drop=True)

In [66]:
# read in 2014 lo_ssc data
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc.pkl")

In [ ]:
print(data["obs"])

         cid         lon        lat                time          z         SA  \
0        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.148984   
1        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149086   
2        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149387   
3        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149889   
4        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -5.255167  31.145790   
...      ...         ...        ...                 ...        ...        ...   
5013  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -24.400000        NaN   
5014  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -33.900000        NaN   
5015  3351.0 -122.428001  47.744000 2014-12-15 17:42:00 -14.900000        NaN   
5016  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -0.620000        NaN   
5017  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -1.600000        NaN   

            CT          DO 

In [ ]:
(data["obs"]["source"] == "ecology_nc").sum()

np.int64(1283)

In [ ]:
data["obs"].columns

Index(['cid', 'lon', 'lat', 'time', 'z', 'SA', 'CT', 'DO', 'NO3', 'Chl',
       'name', 'cruise', 'source', 'NH4', 'PO4 (uM)', 'SiO4 (uM)', 'NO2 (uM)',
       'TA', 'DIC'],
      dtype='object')

In [67]:
# ============================================================
# Prepare the larger observation dataset
# ============================================================

obs = data["obs"].copy()

# Preserve the original row index
obs = obs.reset_index().rename(columns={"index": "obs_index"})

obs["name"] = (
    obs["name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

obs["time"] = pd.to_datetime(
    obs["time"],
    errors="coerce"
)

obs["z"] = pd.to_numeric(
    obs["z"],
    errors="coerce"
)

# Matching keys
obs["match_time"] = obs["time"]
obs["match_z"] = obs["z"].round(0)

# ============================================================
# Prepare the SSM dataset
# ============================================================

ssm = ssm.rename(
    columns={
        "Station": "name",
        "Model_Time": "time",
        "Depth_m": "z",

        # Ecology observational values
        "Temp_C": "obs_CT",
        "Salinity_psu": "obs_SA",
        "DO_mgL": "obs_DO_mgL",
        "NO23N_mgL": "obs_NO3_mgL",
        "NH4N_mgL": "obs_NH4_mgL",
        "Chla_ugL": "obs_Chl",

        # SSM model output
        "Model_Temp_C": "CT",
        "Model_Salinity_psu": "SA",
        "Model_DO_mgL": "DO",
        "Model_NO23N_mgL": "NO3",
        "Model_NH4N_mgL": "NH4",
        "Model_Chla_ugL": "Chl",
    }
)

ssm["name"] = (
    ssm["name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

ssm["time"] = pd.to_datetime(
    ssm["time"],
    errors="coerce"
)

ssm["z"] = pd.to_numeric(
    ssm["z"],
    errors="coerce"
)

# Convert positive-downward SSM depths to negative-downward
ssm["z"] = -ssm["z"].abs()

ssm["match_time"] = ssm["time"]
ssm["match_z"] = ssm["z"].round(0)

# Give every SSM row a unique identifier
ssm = ssm.reset_index(drop=True)
ssm["ssm_row_id"] = ssm.index

# ============================================================
# Convert SSM observation units
# ============================================================

ssm["obs_CT"] = pd.to_numeric(
    ssm["obs_CT"],
    errors="coerce"
)

ssm["obs_SA"] = pd.to_numeric(
    ssm["obs_SA"],
    errors="coerce"
)

ssm["obs_Chl"] = pd.to_numeric(
    ssm["obs_Chl"],
    errors="coerce"
)

# mg O2/L -> µmol O2/L
ssm["obs_DO"] = (
    pd.to_numeric(ssm["obs_DO_mgL"], errors="coerce")
    * 1000
    / 31.998
)

# These conversions assume the spreadsheet reports nutrients as mg N/L
ssm["obs_NO3"] = (
    pd.to_numeric(ssm["obs_NO3_mgL"], errors="coerce")
    * 1000
    / 14.007
)

ssm["obs_NH4"] = (
    pd.to_numeric(ssm["obs_NH4_mgL"], errors="coerce")
    * 1000
    / 14.007
)

# ============================================================
# Convert SSM model output units
# ============================================================

# DO: mg O2/L -> µmol/L
ssm["DO"] = (
    pd.to_numeric(ssm["DO"], errors="coerce")
    * 1000 / 31.998
)

# NO3: mg N/L -> µmol/L
ssm["NO3"] = (
    pd.to_numeric(ssm["NO3"], errors="coerce")
    * 1000 / 14.007
)

# NH4: mg N/L -> µmol/L
ssm["NH4"] = (
    pd.to_numeric(ssm["NH4"], errors="coerce")
    * 1000 / 14.007
)

In [68]:
print(
    ssm[
        [
            "obs_DO_mgL",
            "obs_DO",
            "DO",
            "obs_NO3_mgL",
            "obs_NO3",
            "NO3",
            "obs_NH4_mgL",
            "obs_NH4",
            "NH4",
        ]
    ].head()
)

   obs_DO_mgL  obs_DO          DO  obs_NO3_mgL    obs_NO3        NO3  \
0         NaN     NaN  310.649260     0.374497  26.736427  28.322749   
1         NaN     NaN  273.939654     0.368922  26.338436  30.605400   
2         NaN     NaN  234.453609     0.368908  26.337436  30.210980   
3         NaN     NaN  321.274154     0.325264  23.221502  19.303428   
4         NaN     NaN  309.824489     0.324745  23.184504  21.422719   

   obs_NH4_mgL   obs_NH4       NH4  
0     0.004034  0.287994  1.037441  
1     0.004188  0.298994  0.784688  
2     0.005351  0.381992  0.760547  
3     0.008446  0.602987  1.590272  
4     0.007970  0.568988  1.744906  


In [70]:
TIME_TOLERANCE = pd.Timedelta(hours=3)

obs_candidates = obs.dropna(
    subset=["name", "match_z", "time"]
).copy()

ssm_candidates = ssm.dropna(
    subset=["name", "match_z", "time"]
).copy()

candidate_matches = obs_candidates.merge(
    ssm_candidates,
    on=["name", "match_z"],
    how="inner",
    suffixes=("_large", "_ssm")
)

candidate_matches["time_difference"] = (
    candidate_matches["time_large"]
    - candidate_matches["time_ssm"]
).abs()

candidate_matches = candidate_matches[
    candidate_matches["time_difference"].le(TIME_TOLERANCE)
].copy()

print(f"Candidate rows: {len(candidate_matches):,}")

# Count candidates per observation
match_counts = candidate_matches.groupby("obs_index").size()

# Keep only observations with exactly one candidate
unique_obs = match_counts[match_counts == 1].index

matched = candidate_matches[
    candidate_matches["obs_index"].isin(unique_obs)
].copy()

print(f"Matched rows (unique only): {len(matched):,}")
print(
    f"Observations discarded due to multiple candidates: "
    f"{(match_counts > 1).sum():,}"
)

Candidate rows: 1,075
Matched rows (unique only): 1,049
Observations discarded due to multiple candidates: 13


In [71]:
# ============================================================
# Filter original dataframes
# ============================================================

matched_obs_indices = matched["obs_index"].to_numpy()

data_filtered = {}

for key, df in data.items():

    if not isinstance(df, pd.DataFrame):
        data_filtered[key] = df
        continue

    data_filtered[key] = df.loc[
        df.index.isin(matched_obs_indices)
    ].copy()

    print(
        f"{key}: {len(df):,} -> "
        f"{len(data_filtered[key]):,}"
    )

obs: 5,018 -> 1,049
cas7_t1_x11ab: 5,018 -> 1,049
ssc: 5,018 -> 1,049


In [72]:
# ============================================================
# Build SSM dataframe aligned with filtered observations
# ============================================================

ssm_out = matched[
    [
        "obs_index",
        "time_ssm",
        "name",
        "z_ssm",
        "CT_ssm",
        "SA_ssm",
        "DO_ssm",
        "NO3_ssm",
        "NH4_ssm",
        "Chl_ssm",
    ]
].copy()

ssm_out = ssm_out.rename(
    columns={
        "time_ssm": "time",
        "z_ssm": "z",
        "CT_ssm": "CT",
        "SA_ssm": "SA",
        "DO_ssm": "DO",
        "NO3_ssm": "NO3",
        "NH4_ssm": "NH4",
        "Chl_ssm": "Chl",
    }
)

# Use the original observation index
ssm_out = ssm_out.set_index("obs_index")

# Confirm one SSM row per observation
if not ssm_out.index.is_unique:
    raise ValueError("Duplicate obs_index values remain in ssm_out.")

# Align row order with the filtered observations
ssm_out = ssm_out.reindex(data_filtered["obs"].index)

# Add to dictionary
data_filtered["ssm"] = ssm_out

In [73]:
for key in ["obs", "cas7_t1_x11ab", "ssc", "ssm"]:
    print(
        key,
        len(data_filtered[key]),
        data_filtered[key].index.equals(
            data_filtered["obs"].index
        )
    )

obs 1049 True
cas7_t1_x11ab 1049 True
ssc 1049 True
ssm 1049 True


In [74]:
print(data_filtered['ssm'])

                    time    name      z         CT         SA          DO  \
3694 2014-04-09 08:00:00  KSBP01  -0.85   9.340279  25.815477  309.577591   
3695 2014-04-09 08:00:00  KSBP01  -2.30   9.340279  25.815477  309.577591   
3697 2014-04-22 13:00:00  KSBP01 -99.20   8.950534  29.597321  250.021646   
3698 2014-04-22 13:00:00  KSBP01 -54.50   8.992873  29.495943  254.585114   
3699 2014-04-22 13:00:00  KSBP01 -24.60   9.099450  29.216654  264.979830   
...                  ...     ...    ...        ...        ...         ...   
4865 2014-12-16 11:00:00  MSJN02 -54.60  10.860485  29.869287  220.124919   
4866 2014-12-16 11:00:00  MSJN02 -34.80  10.822799  29.816681  223.187255   
4867 2014-12-16 11:00:00  MSJN02 -15.00  10.700956  29.416845  242.796248   
4868 2014-12-16 11:00:00  MSJN02 -24.80  10.853752  29.754335  227.877331   
4869 2014-12-16 11:00:00  MSJN02  -1.00  10.134727  28.472149  273.677139   

            NO3       NH4       Chl  
3694  24.378615  1.589473  7.170280  

In [75]:
# save the combined data
pd.to_pickle(data_filtered, 'try3_combined_bottle_2014_cas7_t1_x11ab_ssc_ssm.pkl')

In [ ]:
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc_ssm.pkl")

In [ ]:
for k in data:
    print(k, len(data[k]))

obs 1077
cas7_t1_x11ab 1077
ssc 1077
meta 2
ssm 1077


In [ ]:
do_count = data['ssm']['DO'].notna().sum()
no3_count = data['ssm']['NO3'].notna().sum()

print("DO count:", do_count)
print("NO3 count:", no3_count)

DO count: 1077
NO3 count: 1077
